# Lab 12 Solution: Exploring Transformer Layers and Attention Heads

In [2]:
pip -q install transformers torch

Note: you may need to restart the kernel to use updated packages.


## Load `bert-base-uncased`

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, attn_implementation="eager")
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

## 1. Number of layers and attention heads per layer

In [4]:
num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads

print("Transformer layers:", num_layers)
print("Attention heads per layer:", num_heads)

Transformer layers: 12
Attention heads per layer: 12


## 2. Total number of attention heads

In [5]:
total_heads = num_layers * num_heads

print("Total attention heads:", total_heads)

Total attention heads: 144


## 3. Attention weights: third layer, second head

In [6]:
sentence = "Transformers use self-attention to understand language."

inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attention = outputs.attentions[2][0, 1]

print("Tokens:", tokens)
print("Attention shape:", attention.shape)
print("Attention weights:\n", attention)
print("Row sums:", attention.sum(dim=1))

Tokens: ['[CLS]', 'transformers', 'use', 'self', '-', 'attention', 'to', 'understand', 'language', '.', '[SEP]']
Attention shape: torch.Size([11, 11])
Attention weights:
 tensor([[7.3176e-01, 7.9132e-03, 1.0591e-02, 5.5538e-03, 1.6532e-02, 6.8604e-03,
         1.0085e-02, 1.1588e-02, 5.5202e-03, 3.8613e-02, 1.5499e-01],
        [8.8883e-01, 4.7016e-03, 3.4382e-02, 1.7840e-04, 2.7377e-03, 5.0542e-03,
         9.0185e-04, 7.3192e-04, 4.0114e-04, 5.1106e-03, 5.6975e-02],
        [1.4273e-01, 5.6892e-01, 5.4059e-03, 3.5340e-02, 4.1156e-02, 6.7886e-02,
         8.9602e-02, 3.2741e-04, 5.3299e-03, 8.9671e-03, 3.4334e-02],
        [3.3953e-01, 7.3164e-02, 1.5250e-03, 1.6106e-04, 2.0160e-01, 8.1124e-02,
         6.3570e-02, 2.5709e-02, 1.6392e-03, 2.8494e-02, 1.8348e-01],
        [1.4291e-01, 6.9677e-02, 7.1289e-02, 6.3943e-01, 1.2104e-02, 1.2819e-02,
         3.4381e-03, 2.5649e-03, 1.4182e-02, 9.2673e-03, 2.2316e-02],
        [1.1261e-01, 5.4484e-01, 1.1556e-02, 2.0262e-02, 1.1117e-01, 1.021

## 4. What the rows and columns represent

The attention matrix has shape `(sequence_length, sequence_length)`, one row and one column per token (including `[CLS]` and `[SEP]`).

- **Rows** are the *query* tokens: the token that is currently being updated and is "looking" at the sentence.
- **Columns** are the *key* tokens: the tokens being looked at.
- **Entry `[i, j]`** is how much attention token `i` pays to token `j` when building its new representation.
- Each row is passed through softmax, so its values are between 0 and 1 and sum to 1 (see the row sums above).

A large value in row `i`, column `j` means token `j` contributes strongly to the contextual representation of token `i`.